# 01 Probe set
Run 10 probe tasks 5 times on two models and compare.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork (and BRANCH).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
BRANCH = "main"  # change to your branch name if you are working on a branch
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: pick two models

In [ ]:
from kit.llm import call_llm, offline
from kit.prices import cost
import json, re
import pandas as pd

MODEL_A = os.environ.get("PROBE_MODEL_A", "gemini-3.5-flash-lite")
MODEL_B = os.environ.get("PROBE_MODEL_B", "gemini-3.6-flash")
if offline():
    MODEL_A, MODEL_B = "mock-model-a", "mock-model-b"
RUNS = 5
print(MODEL_A, "vs", MODEL_B)

Step 3: write your 10 probes and their pass criteria

A probe is one fixed test question. Its pass criterion is the rule that decides if an answer passed. Edit the list in the cell below: replace the kit's ten examples with ten questions a Quillnote user might ask, each with a word the answer must contain. Write all ten before you run anything below, so the results cannot change your rules.

In [ ]:
# Your 10 probes: one line per test question, written BEFORE you run Step 5.
# Each line has four parts: (id, the question, check type, check value).
#   "must_contain": the answer passes if it includes the check value exactly (capital letters matter).
#   "valid_json":   the answer passes if it is valid JSON. Leave the check value as "".
#   "regex":        advanced, a text pattern. Ask an AI chat to write one if you need it.
# The ten below are the kit's examples. Replace them with questions for your Quillnote help chatbot,
# for example ("p01", "A user asks: How do I get back a note I deleted? Answer in 3 short steps.", "must_contain", "Trash").
MY_PROBES = [
    ("p01", "A user asks: How do I export a note as PDF? Answer in 3 short steps and mention the word Settings.", "must_contain", "Settings"),
    ("p02", "Reply with JSON only, no other text: an object with keys answer and ok, where ok is true.", "valid_json", ""),
    ("p03", "A user asks: Where is dark mode? Answer in one sentence that includes the word steps.", "must_contain", "steps"),
    ("p04", "Classify this message as billing, bug or howto and reply with JSON only like {\"label\": \"bug\"}: The app crashes when I open a big note.", "valid_json", ""),
    ("p05", "Write a one line answer to: Can I add a tag? It must start with the word Mock or Yes.", "regex", "^(Mock|Yes)"),
    ("p06", "Summarise in under 30 words: My notes stopped syncing after the update and I lost an hour of work.", "regex", "^.{1,240}$"),
    ("p07", "A user asks: Can I share a notebook with my team? Include the word open in your answer.", "must_contain", "open"),
    ("p08", "Return JSON only with a key answer holding a short reply to: How do I undo a deleted note?", "valid_json", ""),
    ("p09", "A user writes: Why was I charged twice? Reply politely and include the word save somewhere.", "must_contain", "save"),
    ("p10", "Give a short answer to: Can I search inside images? Do not use the word sorry.", "regex", "^(?![\\s\\S]*[Ss]orry)[\\s\\S]+$"),
]

probes = pd.DataFrame(MY_PROBES, columns=["probe_id", "task", "check_type", "check_value"])
assert len(probes) == 10, f"You have {len(probes)} probes. The assignment asks for exactly 10."
assert set(probes.check_type) <= {"must_contain", "valid_json", "regex"}, "Check type must be must_contain, valid_json or regex."
probes

Step 4: define the pass checks

In [ ]:
def passes(text, check_type, check_value):
    if check_type == "must_contain":
        return check_value in text
    if check_type == "regex":
        return re.search(check_value, text.strip(), re.S) is not None
    if check_type == "valid_json":
        try:
            json.loads(text.strip())
            return True
        except ValueError:
            return False
    raise ValueError(f"Unknown check type {check_type}")

print(passes('{"ok": true}', "valid_json", ""), passes("hi", "must_contain", "bye"))

Step 5: run every probe 5 times on each model

Every call gets a system prompt with the kit's practice help pages (the non-admin pages in `corpus/`, the same loader as `01_first_calls`). So the probes test the help chatbot over its help centre data, not a bare model.

In [ ]:
# The same help pages as 01_first_calls, in a system prompt, so the probes run over the help centre data.
# Admin only pages stay out. About 3,000 input tokens per call, 100 calls: on the free tier,
# call_llm waits and retries if you hit the per minute limit.
def load_help_pages(folder=ROOT / "corpus"):
    pages = []
    for path in sorted(folder.glob("qn-*.md")):
        _, head, body = path.read_text().split("---", 2)
        meta = dict(line.split(": ", 1) for line in head.strip().splitlines())
        if meta["visibility"] != "admin":
            pages.append(f"[{meta['id']}] {meta['title']}\n{body.strip()}")
    return pages

HELP_PAGES = load_help_pages()
SYSTEM = ("You are the help assistant for Quillnote, a note app.\n"
          "Answer only from the help pages below. Name the page id you used, like [qn-02]. "
          "If the pages do not answer the question, say so.\n\n" + "\n\n".join(HELP_PAGES))
print(len(HELP_PAGES), "help pages in the system prompt, about", len(SYSTEM) // 4, "tokens")

rows = []
for model in [MODEL_A, MODEL_B]:
    for p in probes.itertuples():
        for run in range(1, RUNS + 1):
            r = call_llm(p.task, system=SYSTEM, model=model, temperature=0.7, max_output_tokens=300)
            rows.append({"model": model, "probe_id": p.probe_id, "run": run,
                         "check_type": p.check_type, "check_value": p.check_value,
                         "passed": passes(r["text"], p.check_type, p.check_value),
                         "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
                         "stop_reason": r["stop_reason"],
                         "cost_usd": cost(r["input_tokens"], r["output_tokens"], model),
                         "latency_s": r["latency_s"], "text": r["text"][:300]})
runs = pd.DataFrame(rows)
runs.to_csv(OUTPUT / "probe_runs.csv", index=False)  # Stage 2 reads the failed runs from this file
print(len(runs), "calls, saved to", OUTPUT.name + "/probe_runs.csv")

Step 6: pass rate per probe

In [ ]:
runs.pivot_table(index="probe_id", columns="model", values="passed", aggfunc="mean")

Step 7: compare the two models

In [ ]:
summary = runs.groupby("model").agg(pass_rate=("passed", "mean"), cost_per_task=("cost_usd", "mean"),
                                    median_latency_s=("latency_s", "median"), calls=("passed", "size"))
summary["cost_per_task"] = summary["cost_per_task"].round(6)
summary.to_csv(OUTPUT / "probe_summary.csv")  # Stage 4 scores models with these numbers
summary

Step 8: save the files for later stages

`probe_runs.csv` (every run) and `probe_summary.csv` (the table above) are now in `stage-01/`. In Colab, download both from the Files panel: Stage 2 adds your failed probes to its backlog, and Stage 4 scores models with your numbers. Move them into `stage-01/` when you fork the kit in Chapter 3.

Step 9: five predicted weaknesses

Write five weaknesses you expect your product to have. Tie each one to one layer: model, context, harness or surface. Then pick one, show with the run evidence above that it is false, and rewrite it in a narrower form that matches what the runs show.

| # | Prediction, written before the runs | Layer | What the runs showed | Verdict |
| --- | --- | --- | --- | --- |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |

Disproved prediction, rewritten narrower:

Step 10: README

**Decisions:**

**What I measured:**

**What I would do next:**